# Retrieval-Augmented Generation (RAG) with LangChain

## 1. What is RAG?

**Retrieval-Augmented Generation (RAG)** is a pattern where an LLM retrieves relevant information from an external knowledge source before generating an answer.

Instead of asking the model to answer only from its training data, we provide it with useful context at question time:

```text
User question -> Retrieve relevant documents -> Add documents to the prompt -> Generate answer
```

This is useful for private, changing, or domain-specific information such as company policies, product manuals, course notes, and support documentation.

## 2. Why use RAG?

- **More current answers:** the knowledge base can be updated without retraining the model.
- **Private data:** information can remain in your own database or document store.
- **More grounded responses:** the model can answer from retrieved context instead of guessing.
- **Traceability:** retrieved documents can be shown as sources.

RAG does not guarantee truth. If retrieval returns irrelevant or incomplete context, the model may still produce a poor answer.

## 3. The RAG pipeline

A typical LangChain RAG application has two phases.

### Indexing phase

1. Load documents from files, web pages, databases, or APIs.
2. Split large documents into smaller chunks.
3. Convert each chunk into an embedding vector.
4. Store the chunks and vectors in a vector store.

### Retrieval and generation phase

1. Convert the user question into an embedding.
2. Search the vector store for similar chunks.
3. Insert the retrieved chunks into a prompt.
4. Ask the chat model to answer using that context.

```text
Documents -> Splitter -> Embeddings -> Vector store
                                             |
Question -> Retriever -----------------------+
                                             |
                                  Prompt + Chat model
```

## 4. Minimal LangChain example

The following example shows the core idea. Replace the model, embedding model, and vector store with the providers configured in your environment.

```python
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

# Small example knowledge base
 documents = [
    Document(page_content="RAG retrieves relevant context before the LLM answers."),
    Document(page_content="A vector store searches document embeddings by similarity."),
]

# Index the documents
embeddings = OpenAIEmbeddings()
vector_store = Chroma.from_documents(documents, embedding=embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

prompt = ChatPromptTemplate.from_template(
    """Answer the question using only the context below.
If the context does not contain the answer, say that you do not know.

Context:
{context}

Question: {question}
Answer:"""
)

model = ChatOpenAI(model="gpt-4o-mini")

# Convert retrieved documents into text for the prompt
format_docs = lambda docs: "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | model
)

response = rag_chain.invoke("What does RAG do?")
print(response.content)
```

> **Note:** Remove the accidental leading space before `documents` if you copy this example into a Python cell. It should be aligned with `from` and `embeddings`.

## 5. Important LangChain concepts

- **Document loader:** reads source data, such as a PDF, CSV, web page, or text file.
- **Text splitter:** divides long documents into chunks that fit into the model context window.
- **Embedding model:** maps text to vectors so semantically similar text can be searched.
- **Vector store:** stores embeddings and supports similarity search. Examples include Chroma, FAISS, and Pinecone.
- **Retriever:** provides a common interface for fetching relevant documents.
- **Prompt template:** tells the model how to use the retrieved context.
- **RAG chain:** connects retrieval, prompt construction, and generation.

## 6. Common mistakes and how to avoid them

### Chunks are too large or too small

Large chunks may contain too much unrelated text; tiny chunks may lose important context. Start with a moderate `chunk_size` and some `chunk_overlap`, then evaluate retrieval quality.

### Retrieval returns irrelevant documents

Tune the number of results (`k`), use metadata filters, or try a reranker. Better embeddings alone do not fix poor chunking or noisy source data.

### The model ignores the context

Use a clear instruction such as “answer only from the context.” Also inspect the retrieved documents separately before debugging the prompt or model.

### The answer has no citations

A basic RAG chain passes text to the model but does not automatically create reliable citations. Preserve document metadata such as file name and page number, then format those sources explicitly in the prompt or response.

### Secrets are placed in the notebook

Use environment variables or a secret manager for API keys. Do not commit keys into notebooks or source control.

## 7. A useful debugging checklist

When a RAG answer is wrong, test the stages separately:

1. **Source data:** Is the answer actually present in the documents?
2. **Chunks:** Does one chunk contain enough information to answer?
3. **Retrieval:** Are the expected chunks returned for the question?
4. **Prompt:** Is the retrieved context included correctly?
5. **Generation:** Does the model follow the instruction and stay grounded?

## 8. When should I use RAG?

Use RAG when the model needs access to information that is private, frequently changing, too large to place in every prompt, or specific to a business or project. Fine-tuning is usually a better fit for changing the model's style or behavior, while RAG is better for supplying changing factual knowledge.
